# 02 - Feature engineering and selection

**Goal:** turn the findings of notebook 01 into features, and decide which columns to keep.

- Uses the cleaned data saved by `01_eda_cleaning_all.ipynb` (run it first).
- Same rules as 01: `spend_next_90d` is never a feature, and anything learned from the data is fitted on train only.

In [6]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy.stats import spearmanr

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = "../data"
TARGET, SPEND = "repeat_purchase_90d", "spend_next_90d"
BLUE, ORANGE, GREY, RED = "#2f6690", "#d9822b", "#9aa5b1", "#c0392b"

# Same plot style as notebook 01
pio.templates["eda"] = go.layout.Template(layout=dict(
    font=dict(size=13), title=dict(font=dict(size=17)), height=450,
    margin=dict(t=80, l=60, r=30, b=50), hoverlabel=dict(bgcolor="white")))
pio.templates.default = "plotly_white+eda"

# Cleaned data saved at the end of notebook 01 (run 01 first)
df = pd.read_csv(f"{DATA_DIR}/processed/train_clean.csv", parse_dates=["purchase_date"])
test = pd.read_csv(f"{DATA_DIR}/processed/test_clean.csv", parse_dates=["purchase_date"])
print("train:", df.shape, "| test:", test.shape)

train: (6469, 39) | test: (986, 37)


## 1. Feature ideas from notebook 01

| Feature | Idea | Where it came from (01) |
|---|---|---|
| `is_first_purchase` | 1 when `prior_purchase_count` = 0. On a 1st purchase the history columns are empty *by definition* (no history, not unknown): fill them with a constant, and the flag tells the model what that constant means. 1st purchases repeat 33%, returning customers 58%. | Sections 5 and 8 |
| `gap_ratio` = `days_since_previous_purchase / prior_mean_gap_days` | Is the customer late compared with their usual rhythm? Tested in Section 2: weak on its own. | Section 8 (recency and usual gap) |
| `month` | Month of the purchase (one-hot, or sine/cosine). March and November repeat about 45%; January, May and September about 70%. | Section 7 |
| `country`, `island_missing` | From `clientes.csv`. CT5 repeats 39%, CT10 57%; `island` filled 64% vs 52%. Group small countries into "other". | Section 11 |
| `vat_rate`, `margin`, `cost_is_zero`, `discount_share` | Ratios from the invoice columns (Section 3 below). | Section 4 |
| `log1p` of the money columns | Values are very skewed; helps linear models. | Sections 4 and 9 |

**To drop:** `random_noise` (already dropped in 01), `document_series`, the address columns of `clientes.csv` and `customer_type` (Sections 10 and 11 of 01).

## 2. Is the customer late compared with their usual rhythm? (`gap_ratio`)

**Idea:** "60 days since the last purchase" means different things for different clients. For someone who buys every 10 days it is very late; for someone who buys every 90 days it is normal. So we compare each client with **their own habit**:

`gap_ratio` = `days_since_previous_purchase` / `prior_mean_gap_days`

| client | usually buys every | last purchase | gap_ratio | meaning |
|---|---|---|---|---|
| A | 30 days | 15 days ago | 0.5 | early |
| B | 30 days | 30 days ago | 1 | on time |
| C | 30 days | 120 days ago | 4 | very late, maybe loosing the client |

**Expected:** the later the client (higher ratio), the less likely they come back.

Notes: the ratio is empty on 1st and 2nd purchases (no average gap yet). We divide by `prior_mean_gap_days + 1` so a gap of 0 days doesn't break the division.

The test has three parts:
1. Repeat rate per ratio band.
2. Clients with the **same** days since their last purchase, but a different ratio. Does being late still matter?
3. Correlation with the target: the ratio vs the two columns it is made from.

In [7]:
gap_ratio = df["days_since_previous_purchase"] / (df["prior_mean_gap_days"] + 1)
has_ratio = gap_ratio.notna()
print("Rows with a ratio:", has_ratio.sum(), "of", len(df), "(empty on 1st and 2nd purchases)")

# 1. Repeat rate per ratio band
ratio_band = pd.cut(gap_ratio, [0, 0.5, 1, 2, 4, np.inf], right=False,
                    labels=["< 0.5 (early)", "0.5-1", "1-2 (late)", "2-4", "4+ (very late)"])
print("\n1. Repeat rate per gap_ratio band")
display(df.groupby(ratio_band, observed=True)[TARGET].agg(["mean", "size"]).round(3)
          .rename(columns={"mean": "repeat rate", "size": "rows"}).rename_axis("gap_ratio"))

# 2. Same days since the last purchase (rows), on time vs late (columns)
days_band = pd.cut(df["days_since_previous_purchase"], [0, 30, 90, 180, np.inf], right=False,
                   labels=["0-29 days", "30-89 days", "90-179 days", "180+ days"])
late = pd.cut(gap_ratio, [0, 1, 2, np.inf], right=False, labels=["on time (< 1)", "late (1-2)", "very late (2+)"])
print("\n2. Repeat rate: same days since the last purchase, different gap_ratio")
display(df[has_ratio].groupby([days_band, late], observed=True)[TARGET].mean().unstack().round(2)
          .rename_axis(index="days since last purchase", columns="gap_ratio"))
print("Typical usual gap (median prior_mean_gap_days) per group:")
display(df[has_ratio].groupby([days_band, late], observed=True)["prior_mean_gap_days"].median().unstack().round(0)
          .rename_axis(index="days since last purchase", columns="gap_ratio"))

# 3. Correlation with the target, on the same rows
print("\n3. Spearman correlation with the target (same rows):")
for name, values in [("days_since_previous_purchase", df["days_since_previous_purchase"]),
                     ("prior_mean_gap_days", df["prior_mean_gap_days"]), ("gap_ratio", gap_ratio)]:
    print(f"  {name:30s} {spearmanr(values[has_ratio], df.loc[has_ratio, TARGET], nan_policy='omit')[0]:.2f}")

Rows with a ratio: 5056 of 6469 (empty on 1st and 2nd purchases)

1. Repeat rate per gap_ratio band


,repeat rate,rows
gap_ratio,,
< 0.5 (early),0.632,1752
0.5-1,0.627,1229
1-2 (late),0.564,1313
2-4,0.571,566
4+ (very late),0.423,196



2. Repeat rate: same days since the last purchase, different gap_ratio


gap_ratio,on time (< 1),late (1-2),very late (2+)
days since last purchase,,,
0-29 days,0.72,0.96,0.86
30-89 days,0.63,0.79,0.79
90-179 days,0.31,0.43,0.61
180+ days,0.22,0.39,0.38


Typical usual gap (median prior_mean_gap_days) per group:


gap_ratio,on time (< 1),late (1-2),very late (2+)
days since last purchase,,,
0-29 days,51.0,11.0,7.0
30-89 days,84.0,48.0,26.0
90-179 days,145.0,85.0,45.0
180+ days,370.0,166.0,86.0



3. Spearman correlation with the target (same rows):
  days_since_previous_purchase   -0.31
  prior_mean_gap_days            -0.42
  gap_ratio                      -0.07


### Key insight - Section 2

**Found**
- **Overall, the idea seems to hold:** very late clients (ratio 4+) repeat 42%, early ones about 63%.
- **But with the same days since the last purchase, late clients repeat *more*, not less.** For example, at 90-179 days: on time 31%, very late 61%.
- **Why:** with the same days since the last purchase, a "late" client is one whose usual gap is **short**, so they normally buy often. Frequent buyers come back (Section 8 of 01), even when they are a bit late. The second table shows this: the usual gap is much shorter in the "late" columns.
- So on its own the ratio is weak (correlation -0.07). The two columns it is made from are much stronger: the usual gap (-0.42) and the days since the last purchase (-0.31).

**For feature engineering**
- Keep `days_since_previous_purchase` and `prior_mean_gap_days` as separate features: that's where the signal is. Tree models can combine them on their own.
- `gap_ratio` is low priority: add it only if it improves the score in validation.

## 3. Ratios from the invoice columns

- **Margin** = (net value - cost) / net value: the share of the sale that is profit.
- **VAT rate** = tax / net value.

Each one is compared with the target. Repeat rate = share of purchases followed by another purchase within 90 days (overall 0.56).

In [8]:
# Two ratios built from the invoice columns, each checked against the target
margin = (df["Total Liquido"] - df["Custo"]) / df["Total Liquido"]   # share of the net value that is profit (1 = cost 0, < 0 = loss)
tax_rate = df["Total Imp_"] / df["Total Liquido"]                     # VAT rate of the invoice

print("Cost = 0:", (df["Custo"] == 0).sum(), "| cost > net value (loss):", (df["Custo"] > df["Total Liquido"]).sum())

# Repeat rate per VAT band: the 4 most common rates, everything else (mixed-rate invoices) grouped as "other"
vat_band = tax_rate.round(2)
vat_band = vat_band.where(vat_band.isin([0.0, 0.13, 0.22, 0.23]), "other").astype(str)
display(df.groupby(vat_band)[TARGET].agg(["mean", "size"]).round(3)
          .rename(columns={"mean": "repeat rate", "size": "rows"}).rename_axis("VAT rate"))

# Repeat rate per margin band
margin_band = pd.cut(margin, [-np.inf, 0, 0.2, 0.3, 0.4, 0.999, 1],
                     labels=["loss (<= 0)", "0-20%", "20-30%", "30-40%", "40-99.9%", "100% (cost = 0)"])
display(df.groupby(margin_band, observed=True)[TARGET].agg(["mean", "size"]).round(3)
          .rename(columns={"mean": "repeat rate", "size": "rows"}).rename_axis("margin"))

print("Overall repeat rate:", round(df[TARGET].mean(), 3))

Cost = 0: 486 | cost > net value (loss): 39


,repeat rate,rows
VAT rate,,
0.0,0.390,213
0.13,0.701,221
0.22,0.678,115
0.23,0.565,5641
other,0.416,279


,repeat rate,rows
margin,,
loss (<= 0),0.756,41
0-20%,0.640,114
20-30%,0.606,381
30-40%,0.427,1362
40-99.9%,0.582,4085
100% (cost = 0),0.677,486


Overall repeat rate: 0.56


### Key insight - Section 3

**Found**
- **VAT rate is related to the target.** 0% VAT repeats 39%, while 13% and 22% VAT repeat about 70% (overall 56%). 193 of the 213 rows at 0% VAT belong to country CT5, so they are probably foreign customers.
- **Margin is related to the target, but not in a straight line.** The 30-40% band repeats only 43%; low margins and losses repeat more.
- **486 sales (7.5%) have cost = 0.** They are mostly short invoices, more common in 2018-2020, and some are above 3,000 EUR, so they are not free samples. Possible reasons: in-house products or services, or cost not recorded. Either way, their 100% margin is not reliable.
- 39 sales are at a loss. Plausible (discounts), so they are kept.

**For feature engineering**
- Create `vat_rate` (or VAT band), `margin` and `discount_share` (discounts / gross).
- Add a `cost_is_zero` flag and set the margin of those rows to NaN, to be imputed in the pipeline.
- `vat_rate = 0` and country CT5 carry the same information: keep only one of them.

## 4. Which columns are related to `repeat_purchase_90d`?

Spearman correlation of each numeric column with the target. Spearman only sees relations that go in one direction (always up or always down), so a column with a curved effect (like margin) can look weak here and still be useful for a tree model.

In [9]:
num_cols = df.select_dtypes("number").columns.drop(["ID", "customer_id", TARGET, SPEND], errors="ignore")
corr_target = pd.Series({c: spearmanr(df[c], df[TARGET], nan_policy="omit")[0] for c in num_cols}).sort_values()

colors = [BLUE if v > 0 else ORANGE for v in corr_target]
fig = go.Figure(go.Bar(x=corr_target.values, y=corr_target.index, orientation="h", marker_color=colors,
                       hovertemplate="%{y}: %{x:.3f}<extra></extra>"))
fig.add_vline(x=0, line_color="black", line_width=1)
fig.update_layout(title="Spearman correlation with repeat_purchase_90d", xaxis_title="correlation", height=800)
fig.show()

### Key insight - Section 4

**Found**
- The signal is in the **customer's history**: purchases in recent windows, recency and the average gap (correlation 0.35-0.44).
- The current purchase barely matters (below 0.1), except line discounts (0.2).

**For feature selection**
- Keep the history columns: they carry most of the signal.
- The current-purchase columns are candidates to drop. Confirm with a model-based method (e.g. permutation importance), since Spearman misses curved effects.

## 5. Which predictors are near copies of each other?

Two columns that carry the same information add nothing for the model, and they make linear models unstable.

In [10]:
corr_matrix = df[num_cols].corr(method="spearman")
fig = px.imshow(corr_matrix, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="equal")
fig.update_layout(title="Spearman correlation between numeric predictors", height=900, width=1000)
fig.show()

pairs = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)).stack()
print("Pairs with |rho| > 0.9:")
print(pairs[pairs.abs() > 0.9].sort_values(ascending=False).round(3).to_string())

Pairs with |rho| > 0.9:
Total a Pagar             Total Liquido               1.000
Total Bruto               Total Liquido               0.991
Total a Pagar             Total Bruto                 0.991
purchase_count_30d        spend_30d                   0.965
prior_std_purchase_value  prior_max_purchase_value    0.935
purchase_count_90d        spend_90d                   0.909
prior_purchase_count      prior_total_spend           0.903
spend_180d                spend_365d                  0.900


### Key insight - Section 5

**Found** - several pairs are near copies (correlation > 0.9):
- the money columns of the invoice (`Total a Pagar`, `Total Liquido`, `Total Bruto`),
- count vs spend in the same window (30 and 90 days),
- `spend_180d` vs `spend_365d`,
- `prior_purchase_count` vs `prior_total_spend`,
- `prior_std_purchase_value` vs `prior_max_purchase_value`.

**For feature selection**
- Keep one money column (e.g. `Total Liquido`) and drop the others.
- From each other pair, keep the one more related to the target (Section 4), or let the selection method decide.

## 6. Which columns are related to `spend_next_90d`?

Only customers who came back (target = 1) have a spend value, so the correlation uses those rows. It is shown next to the correlation with repeat from Section 4.

In [11]:
returned = df[(df[TARGET] == 1) & df[SPEND].notna()]
corr_spend = pd.Series({c: spearmanr(returned[c], returned[SPEND], nan_policy="omit")[0] for c in num_cols})
both = pd.DataFrame({"repeat (classification)": corr_target, "spend (regression, returners)": corr_spend})
both = both.reindex(both["spend (regression, returners)"].abs().sort_values().index)

fig = go.Figure()
for name, color in [("repeat (classification)", GREY), ("spend (regression, returners)", BLUE)]:
    fig.add_bar(x=both[name], y=both.index, orientation="h", name=name, marker_color=color)
fig.add_vline(x=0, line_color="black", line_width=1)
fig.update_layout(title="Same predictors, two targets (Spearman)", barmode="group", height=900)
fig.show()

### Key insight - Section 6

**Found**
- Spend depends mostly on **past spend**: `spend_365d` (0.60), `spend_180d` (0.56), `prior_total_spend` (0.52).
- The current purchase value matters a bit more than for repeat (0.16 vs 0.05), but it is still secondary.
- The features rank in a similar order for both targets.

**For feature selection**
- Use the same feature set for both models: simpler, and the two predictions stay consistent.

## Summary

**Create**
- `is_first_purchase` (+ fill the "no history" NaN with a constant)
- `gap_ratio` only if it improves validation (Section 2: weak on its own)
- `month`, `country` (grouped), `island_missing`
- `vat_rate` (or `country`, not both), `margin`, `cost_is_zero`, `discount_share`

**Drop**
- `random_noise` (done in 01), `document_series`, the address columns of `clientes.csv`, `customer_type`
- the redundant money columns (keep one, e.g. `Total Liquido`)
- one column of each near-copy pair, unless the selection method shows both help

**Next:** build these features inside the sklearn `Pipeline` (fitted on train only) and confirm the selection with time-based validation.